<a href="https://colab.research.google.com/github/OSGeoLabBp/tutorials/blob/master/hungarian/pointcloud/eigen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sajátértékek használata a pontosztályozásban

Egy pont környezetében lévő pontokból, a koordinátakülönbségből készítsünk egy variancia-covariancia mátrixot, Ennek a mátrixnak a sajátértékei a pontok térbeli elrendezésére jellemző.

![jellemzok](https://camo.githubusercontent.com/5eebf55034fff87614bf865af35157fe1923e6983a0a967ec66a9858ca61b493/68747470733a2f2f6769746875622e636f6d2f4f5347656f4c616242702f7475746f7269616c732f626c6f622f6d61737465722f68756e67617269616e2f6d616368696e655f6c6561726e696e672f696d616765732f73616a6174657274656b656b2e706e673f7261773d74727565)

In [1]:
!pip install -q plotly

In [2]:
import numpy as np
from numpy import linalg as la
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [3]:
def pc_show(pts, width=800, height=600, extra=None):
    """ Pontfelhő megjelenítése
        az extra paraméter xyz tömbök listája, a pontokat egyenessel köti össze
    """
    # megjelenítés
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode='markers', marker=dict(size=1, color='blue',))
    )
    if extra is not None:
        for l in extra:
            fig.add_trace(go.Scatter3d(x=l[:,0], y=l[:,1], z=l[:,2], mode='lines', line=dict(color='red', width=3)))
    # testreszabás
    fig.update_layout(width=width, height=height,
        scene=dict(aspectmode='data',),
        margin=dict(l=0, r=0, b=0, t=0)
    )
    fig.show()

In [4]:
max_coo = 2
n_p = 500
noise = 0.05
dist_limit = 0.5
omnivariance = {}
anisotropy = {}
planarity = {}
surface_variation = {}
sphericity = {}
linearity = {}

In [5]:
east0 = max_coo / 2
north0 = max_coo / 2
elev0 = max_coo / 2
enz0 = np.array([east0, north0, elev0])

Véletlen pontok

In [6]:
east = np.random.rand(n_p) * max_coo    # generate random coords
north = np.random.rand(n_p) * max_coo
elev = np.random.rand(n_p) * max_coo
enz = np.c_[east, north, elev]

In [7]:
dists = np.sqrt(np.square(east - east0) + np.square(north - north0) + np.square(elev - elev0))
enz_dist = enz[dists < dist_limit] - enz0
print(enz_dist.shape)

(31, 3)


In [8]:
cov = np.matmul(enz_dist.T, enz_dist)
eigenvalues, eigenvectors = la.eig(cov)
l1, l2, l3 = np.sort(eigenvalues)[::-1]
extras = []
for i in range(3):
    extras.append(np.array([[east0, north0, elev0], [elev0+eigenvectors[i][0], enz0[1]+eigenvectors[i][1], enz0[2]+eigenvectors[i][2]]]))

In [9]:
omnivariance['rand'] = (l1 * l2 * l3)** (1/3)
anisotropy['rand'] = (l1 - l3) / l1
planarity['rand'] = (l2 -l3) / l1
surface_variation['rand'] = l3 / (l1 + l2 + l3)
sphericity['rand'] = l3 / l1
linearity["rand"] = (l1 - l2) / l1

In [10]:
pc_show(enz, extra=extras)

Síkra eső pontok generálása

In [11]:
param_0 = np.array([0.27735, 0.30829, 0.11547, 0])
param_0[3] = -np.dot(param_0[:3], enz0)
east = np.random.rand(n_p) * max_coo    # generate random coords
north = np.random.rand(n_p) * max_coo
elev = -(param_0[0] * east + param_0[1] * north + param_0[3]) / param_0[2]
enz = np.c_[east, north, elev]
enz += np.random.rand(*enz.shape) * noise / 3   # add noise

In [12]:
dists = np.sqrt(np.square(east - east0) + np.square(north - north0) + np.square(elev - elev0))
enz_dist = enz[dists < dist_limit] - enz0
print(enz_dist.shape)

(30, 3)


In [13]:
cov = np.matmul(enz_dist.T, enz_dist)
eigenvalues, eigenvectors = la.eig(cov)
l1, l2, l3 = np.sort(eigenvalues)[::-1]
extras = []
for i in range(3):
    extras.append(np.array([[east0, north0, elev0], [elev0+eigenvectors[i][0], enz0[1]+eigenvectors[i][1], enz0[2]+eigenvectors[i][2]]]))

In [14]:
omnivariance['plane'] = (l1 * l2 * l3)** (1/3)
anisotropy['plane'] = (l1 - l3) / l1
planarity['plane'] = (l2 -l3) / l1
surface_variation['plane'] = l3 / (l1 + l2 + l3)
sphericity['plane'] = l3 / l1
linearity["plane"] = (l1 - l2) / l1

In [15]:
pc_show(enz, extra=extras)

3D vonalra eső pontok generálása

In [17]:
param_0 = np.array([max_coo / 2, max_coo / 2, max_coo / 2, 0.4356, 0.3245, 0.8396])
t = np.random.rand(n_p) * max_coo - max_coo / 2
east = param_0[0] + t * param_0[3]
north = param_0[1] + t * param_0[4]
elev = param_0[2] + t * param_0[5]
enz = np.c_[east, north, elev]
enz += np.random.rand(*enz.shape) * noise / 3   # add noise

In [18]:
dists = np.sqrt(np.square(east - east0) + np.square(north - north0) + np.square(elev - elev0))
enz_dist = enz[dists < dist_limit] - enz0
print(enz_dist.shape)

(239, 3)


In [19]:
cov = np.matmul(enz_dist.T, enz_dist)
eigenvalues, eigenvectors = la.eig(cov)
l1, l2, l3 = np.sort(eigenvalues)[::-1]
extras = []
for i in range(3):
    extras.append(np.array([[east0, north0, elev0], [elev0+eigenvectors[i][0], enz0[1]+eigenvectors[i][1], enz0[2]+eigenvectors[i][2]]]))

In [20]:
omnivariance['line'] = (l1 * l2 * l3)** (1/3)
anisotropy['line'] = (l1 - l3) / l1
planarity['line'] = (l2 -
                     l3) / l1
surface_variation['line'] = l3 / (l1 + l2 + l3)
sphericity['line'] = l3 / l1
linearity["line"] = (l1 - l2) / l1

In [21]:
pc_show(enz, extra=extras)

In [22]:
print("type     omnivariance anisotropy planarity surface_vari sphericity linearity")
for typ in omnivariance:
    print(f"{typ:8s} {omnivariance[typ]:10.5f} {anisotropy[typ]:10.5f}, {planarity[typ]:10.5f} {surface_variation[typ]:10.5f} {sphericity[typ]:10.5f} {linearity[typ]:10.5f}")

type     omnivariance anisotropy planarity surface_vari sphericity linearity
rand        1.66865    0.22037,    0.10671    0.29244    0.77963    0.11366
plane       0.26850    0.99760,    0.59502    0.00150    0.00240    0.40259
line        0.10499    0.99967,    0.00035    0.00033    0.00033    0.99932
